# 4. Honest evaluation

The goal of this project is one honest sentence:

> "WER on my voice went from X% to Y%, while WER on general speech changed by only Z%."

Getting there honestly takes four rules:

1. **Test on sentences the model never trained on, and look at the test set once,** after every decision has been made.
2. **Make every decision on the validation set:** the learning rate, the number of epochs, LoRA or full fine-tuning.
3. **Compare like with like:** the same test sentences, the same normalization, and the same model before fine-tuning.
4. **Report the cost and the uncertainty:** measure general speech (LibriSpeech) before and after, and say how sure you are of the improvement.

The plan:

1. **Part 1, experiments.** Train five variants (three learning rates, a frozen encoder, and LoRA) and compare them on the validation set.
2. **Part 2, the choice.** Pick the winner with a rule you write down *before* you see the results.
3. **Part 3, the test.** Evaluate the chosen model on the test set, once: overall WER, WER per category, and term recall for terms it trained on and terms it never saw.
4. **Part 4, forgetting.** Compare WER on the 73 LibriSpeech clips before and after.
5. **Part 5, uncertainty.** A bootstrap confidence interval for the improvement.
6. **Part 6, the write-up.** Charts, numbers, and your story.

Part 1 trains five models, which takes about 70 minutes on your CPU with the laptop plugged in (on battery, Windows slows the processor down, and it can take several times longer). It saves after each one, so you can stop and continue later without losing finished runs.

## Three ways to fine-tune

### Full fine-tuning

This is what you did in notebook 03: every one of the 37.8 million weights can change. It's the most flexible option, but every fine-tuned version is a full copy of the model (150 MB), and with little data, the model can drift far from the original.

### Freezing the encoder

Whisper's two halves do different jobs. The **encoder** turns sound into representations, so it handles acoustics: your accent, your microphone, your room. The **decoder** turns those representations into text, so it works like a language model: which words and spellings are likely. Your vocabulary problem (writing "Laura" for "LoRA") is mostly a decoder problem.

Freezing the encoder means only the decoder's weights train. Every weight tensor in PyTorch has a flag, **`requires_grad`**. Set it to `False`, and `backward()` computes no gradient for that weight, so the optimizer can't change it. Training gets faster (no gradients to compute for the encoder; about 40% faster per step on your laptop), and the acoustic half can't be damaged. The trade-off is that the model can't adapt to how you sound, only to what you say.

- `module.requires_grad_(False)` sets the flag on every weight inside a module, so `model.model.encoder.requires_grad_(False)` freezes the whole encoder. (`model.model` is the encoder-decoder inside `WhisperForConditionalGeneration`; the outer object adds the final layer that produces logits.)
- The optimizer should get only the weights that still train: `[p for p in model.parameters() if p.requires_grad]`.

### LoRA (low-rank adaptation)

Inside each attention layer are weight matrices, like the 384 × 384 "query" matrix `W`. Full fine-tuning learns a change `ΔW` with all 147,456 of its numbers. LoRA's insight is that the change needed to adapt a model is usually simple, so it can be written as the product of two thin matrices:

    ΔW = B · A      where A has shape (r, 384) and B has shape (384, r), with a small rank r such as 32

With r = 32, `A` and `B` hold 2 × 32 × 384 = 24,576 numbers instead of 147,456. `W` stays frozen, and only `A` and `B` train. The layer computes `W·x + (alpha / r) · B·A·x`, where `alpha` scales how strong the change is. `B` starts at zero, so at the start the model is exactly the original.

Why it matters, especially for a product like Flow:

- **A small fraction trains.** On whisper-tiny.en with r = 32 on the query and value matrices, 589,824 weights train: 1.5% of the model.
- **Small files.** One user's adapter is about 2.4 MB instead of 150 MB, so you could store one per user and load the right one for each request.
- **Less forgetting.** The original weights never change, and the change is constrained to be simple.
- **No extra latency.** After training, you can **merge** `B·A` into `W`, and the model runs exactly as fast as before.

LoRA usually needs a higher learning rate than full fine-tuning, because a few weights have to carry the whole change. We use 1e-3.

### The PEFT library

Hugging Face's **PEFT** (parameter-efficient fine-tuning) library adds LoRA to a model:

    from peft import LoraConfig, get_peft_model
    config = LoraConfig(r=32, lora_alpha=64, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none")
    model = get_peft_model(model, config)

- `r`: the rank of the update.
- `lora_alpha`: the scale; the update is multiplied by `lora_alpha / r`, here 2.
- `target_modules`: the names of the layers that get adapters. In Whisper, `q_proj` and `v_proj` are the query and value matrices of every attention layer: 24 of them in tiny, across the encoder's self-attention, the decoder's self-attention, and the decoder's cross-attention.
- `lora_dropout`: dropout on the adapter's input during training, a mild guard against overfitting.
- `bias="none"`: the bias vectors stay frozen too.

`get_peft_model` returns a `PeftModel` that wraps your model. You use it like the original: calling it with `labels` returns a loss, and `.generate()` works. `model.print_trainable_parameters()` prints how much of it trains, and `model.merge_and_unload()` merges the adapters into the weights and returns a plain Whisper model.

## Setup

This cell imports your functions from `whisper_ft.py` (from notebooks 02 and 03), loads every split, prepares the training examples, and loads the baseline from notebook 02.

- `librispeech` is a list of `(waveform, transcript)` tuples. `[audio for audio, _ in librispeech]` unpacks each tuple and keeps only the waveform; `_` is the usual name for a value you don't need.
- **`SMOKE_TEST`**: set it to `True` to run the whole notebook on a few recordings with 1 epoch per experiment, in about 10 minutes, to catch bugs before the real run. Smoke-test results go to separate files, so they never mix with real ones.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import time

import jiwer
import matplotlib.pyplot as plt
import numpy as np
import torch
import transformers
from matplotlib.ticker import PercentFormatter
from peft import LoraConfig, get_peft_model
from torch.utils.data import DataLoader
from transformers import WhisperForConditionalGeneration

from whisper_ft import (
    MODEL_NAME,
    MODELS_DIR,
    RESULTS_DIR,
    collate,
    contains_term,
    corpus_wer,
    load_json,
    load_librispeech,
    load_metadata,
    load_splits,
    load_vocabulary,
    prepare_example,
    processor,
    read_wav,
    rows_for_split,
    save_json,
    term_recall,
    train_one_epoch,
    transcribe,
)

transformers.logging.set_verbosity_error()
transformers.logging.disable_progress_bar()
SMOKE_TEST = False
SEED = 0

rows = load_metadata()
splits = load_splits()
train_rows = rows_for_split(rows, splits, "train")
val_rows = rows_for_split(rows, splits, "val")
test_rows = rows_for_split(rows, splits, "test")
librispeech = load_librispeech()
if SMOKE_TEST:
    train_rows, val_rows, test_rows, librispeech = train_rows[:16], val_rows[:8], test_rows[:8], librispeech[:8]

train_examples = [prepare_example(row) for row in train_rows]
val_refs = [row["text"] for row in val_rows]
val_audios = [read_wav(row["path"]) for row in val_rows]
libri_audios = [audio for audio, _ in librispeech]
libri_refs = [text for _, text in librispeech]
vocabulary = load_vocabulary()
baseline = load_json(RESULTS_DIR / "baseline.json")

print(f"{len(train_rows)} training, {len(val_rows)} validation, and {len(test_rows)} test recordings; "
      f"{len(librispeech)} LibriSpeech clips")

## Part 1: the experiments

### TODO(you): write `make_model(method)`

Every experiment starts from a fresh whisper-tiny.en, prepared one of three ways.

**Input:** `method`, a `str`: `"full"`, `"frozen_encoder"`, or `"lora"`.

**Output:** the model, ready to train:
- `"full"`: load it and return it unchanged.
- `"frozen_encoder"`: load it, then freeze `model.model.encoder` with `requires_grad_(False)`.
- `"lora"`: load it, then wrap it with `get_peft_model`, using the `LoraConfig` shown above.

Load the model with `WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)`.

`count_trainable(model)` (given) adds up the sizes of all trainable weight tensors: `p.numel()` is the number of values in tensor `p`.

In [ ]:
# TODO(you): a fresh model, prepared for full fine-tuning, a frozen encoder, or LoRA.
def make_model(method):
    ...


def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
counts = {}
for method in ["full", "frozen_encoder", "lora"]:
    m = make_model(method)
    counts[method] = count_trainable(m)
    print(f"{method:>15}: {counts[method]:>11,} trainable weights")
    if method == "frozen_encoder":
        assert not any(p.requires_grad for p in m.model.encoder.parameters()), "no encoder weight should train"
        assert all(p.requires_grad for p in m.model.decoder.parameters()), "every decoder weight should train"
    if method == "lora":
        assert all("lora_" in name for name, p in m.named_parameters() if p.requires_grad), "only the LoRA weights should train"
    del m
assert counts["full"] == 37_760_256, "full fine-tuning trains every weight"
assert counts["frozen_encoder"] == 37_760_256 - 8_208_384, "the encoder has 8,208,384 weights"
assert counts["lora"] == 589_824
print("Correct.")

### The experiment runner

`run_experiment` (below, provided) trains one variant and records the validation WER after every epoch. It uses your `collate` and `train_one_epoch`, so it's your notebook 03 loop with some bookkeeping added:

- **It keeps the best epoch.** Whenever the validation WER reaches a new low, it copies the weights into memory: `model.state_dict()` returns a dict that maps each weight's name to its tensor, and `.detach().clone()` copies each tensor. At the end, it loads the best copy back with `model.load_state_dict(...)`. This is **early stopping**, done after the fact: you train for 5 epochs but keep whichever epoch did best on validation. If no epoch beats the starting point, it keeps the original model.
- **It measures forgetting.** With the best weights restored, it computes WER on the LibriSpeech clips.
- **It saves everything:** the model to `models/<name>/`, and the numbers to `results/experiments.json`. Experiments already in that file are skipped, so you can stop the run and continue it later.

For LoRA, the best adapter is merged into the base model before saving, so every saved model loads the same way.

In [ ]:
EXPERIMENTS_FILE = RESULTS_DIR / ("experiments_smoke.json" if SMOKE_TEST else "experiments.json")


def run_experiment(name, method, learning_rate, epochs=5, batch_size=8):
    torch.manual_seed(SEED)
    model = make_model(method)
    trainable = count_trainable(model)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=learning_rate)
    loader = DataLoader(train_examples, batch_size=batch_size, shuffle=True, collate_fn=collate)

    model.eval()
    history = [{"epoch": 0, "train_loss": None, "val_wer": corpus_wer(val_refs, transcribe(model, val_audios))}]
    best_wer, best_epoch, best_state = history[0]["val_wer"], 0, None
    start = time.perf_counter()
    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, loader, optimizer)
        model.eval()
        val_wer = corpus_wer(val_refs, transcribe(model, val_audios))
        history.append({"epoch": epoch, "train_loss": train_loss, "val_wer": val_wer})
        print(f"  epoch {epoch}: train loss {train_loss:.3f}, val WER {val_wer:.1%}")
        if val_wer < best_wer:
            best_wer, best_epoch = val_wer, epoch
            best_state = {key: value.detach().clone() for key, value in model.state_dict().items()}
    minutes = (time.perf_counter() - start) / 60

    if best_state is None:
        model = make_model(method)
    else:
        model.load_state_dict(best_state)
    if method == "lora":
        model = model.merge_and_unload()
    model.eval()
    libri_wer = corpus_wer(libri_refs, transcribe(model, libri_audios))
    model.save_pretrained(MODELS_DIR / name)
    return {"name": name, "method": method, "learning_rate": learning_rate, "epochs": epochs,
            "trainable_weights": trainable, "best_epoch": best_epoch, "best_val_wer": best_wer,
            "librispeech_wer": libri_wer, "minutes": minutes, "history": history}

### The experiments

Five experiments, 5 epochs each:

| name | method | learning rate | the question it answers |
|---|---|---|---|
| `full_lr1e-5` | full | 1e-5 | your notebook 03 setup |
| `full_lr3e-5` | full | 3e-5 | do bigger steps help? |
| `full_lr1e-4` | full | 1e-4 | what does "too big" look like? |
| `frozen_encoder_lr3e-5` | frozen encoder | 3e-5 | is training the decoder alone enough? |
| `lora_r32_lr1e-3` | LoRA | 1e-3 | can 1.5% of the weights do the job? |

The number of epochs is an experiment too: each run records the validation WER after every epoch, so you'll see where each one peaks.

Each experiment is a tuple `(name, method, learning_rate)`, and the loop unpacks it: `for name, method, learning_rate in EXPERIMENTS:`.

### TODO(you): write down your selection rule first

Before you run anything, decide how you'll pick the winner, and write it in the next cell. If you decide after seeing the results, it's tempting to choose whichever rule makes your favorite look best. Writing the rule down first is called **pre-registration**, and scientists do it for the same reason.

A reasonable rule: "Lowest validation WER. If two are within 1 percentage point, prefer the one with the lower LibriSpeech WER, then the one with fewer trainable weights." Change it if you disagree, but decide now.

**My selection rule:** *(write it here before running the experiments)*

In [ ]:
EXPERIMENTS = [
    ("full_lr1e-5", "full", 1e-5),
    ("full_lr3e-5", "full", 3e-5),
    ("full_lr1e-4", "full", 1e-4),
    ("frozen_encoder_lr3e-5", "frozen_encoder", 3e-5),
    ("lora_r32_lr1e-3", "lora", 1e-3),
]

results = load_json(EXPERIMENTS_FILE) if EXPERIMENTS_FILE.exists() else {}
for name, method, learning_rate in EXPERIMENTS:
    if name in results:
        print(f"{name}: already done, skipping")
        continue
    print(f"{name}: training ({method}, learning rate {learning_rate})")
    results[name] = run_experiment(name, method, learning_rate, epochs=1 if SMOKE_TEST else 5)
    save_json(results, EXPERIMENTS_FILE)
    r = results[name]
    print(f"{name}: best val WER {r['best_val_wer']:.1%} at epoch {r['best_epoch']}, "
          f"LibriSpeech WER {r['librispeech_wer']:.1%}, {r['minutes']:.0f} min\n")

In [ ]:
print(f"{'experiment':24} {'trainable':>12} {'best epoch':>10} {'val WER':>8} {'LibriSpeech':>11}")
for r in results.values():
    print(f"{r['name']:24} {r['trainable_weights']:>12,} {r['best_epoch']:>10} {r['best_val_wer']:>8.1%} {r['librispeech_wer']:>11.1%}")

plt.figure(figsize=(8, 4))
for r in results.values():
    plt.plot([h["epoch"] for h in r["history"]], [h["val_wer"] for h in r["history"]], marker="o", label=r["name"])
plt.xlabel("epoch (0 = before fine-tuning)")
plt.ylabel("validation WER")
plt.gca().yaxis.set_major_formatter(PercentFormatter(1.0))
plt.legend()
plt.title("Validation WER per epoch")
plt.show()

## Part 2: choose the winner

### TODO(you): apply your rule

Set `best_name` to the name of the experiment your rule picks, using only the validation and LibriSpeech numbers above. `results` is a dict that maps each experiment's name to its result dict.

If your rule is just "lowest validation WER", `min(results.values(), key=lambda r: r["best_val_wer"])` returns the winning result dict: `min` with a `key` function returns the item whose key is smallest. Its `"name"` is the name you want.

In [ ]:
# TODO(you): the experiment your selection rule picks.
best_name = ...

best = results[best_name]
print(f"Chosen: {best_name} (best epoch {best['best_epoch']}, val WER {best['best_val_wer']:.1%})")

In [ ]:
assert best_name in results, "best_name should be one of the experiment names"
print("Now, and only now, you look at the test set.")

## Part 3: the test set, once

Load the chosen model from `models/<best_name>/` and evaluate it on the test set exactly the way you evaluated the baseline in notebook 02: the same sentences, the same `transcribe`, the same `corpus_wer`. The baseline's test transcripts are saved in `results/baseline.json`, so only the fine-tuned model needs transcribing.

### Seen and unseen terms

Split your key terms by whether they appear in any **training** sentence:
- **Seen terms** appear in at least one training sentence, so fine-tuning had a chance to learn them.
- **Unseen terms** appear in test sentences but in no training sentence. Any improvement on these comes from something more general, like adapting to your voice.

### TODO(you): the test evaluation

1. Load the model: `WhisperForConditionalGeneration.from_pretrained(MODELS_DIR / best_name).eval()`.
2. Transcribe the test audio into `tuned_hyps`. The baseline transcripts are ready in `base_hyps`, in the same order as `test_rows`.
3. Compute `base_wer` and `tuned_wer` with `corpus_wer`.
4. Compute the WER per category for both models, as two dicts (as in notebook 02; a small helper function that takes a list of hypotheses saves writing it twice).
5. Build `seen_terms` and `unseen_terms` (two lists of str): the vocabulary terms that appear in at least one test reference, split by whether they also appear in at least one training sentence (`train_texts`). Use `contains_term`.
6. Fill the `recall` dict with `term_recall` for both models, on the seen terms and on the unseen terms.

In [ ]:
test_refs = [row["text"] for row in test_rows]
test_audios = [read_wav(row["path"]) for row in test_rows]
base_hyps = [baseline["tiny_test_hypotheses"][row["file"]] for row in test_rows]
train_texts = {row["text"] for row in train_rows}

# TODO(you): evaluate the chosen model on the test set.
tuned_model = ...
tuned_hyps = ...
base_wer, tuned_wer = ..., ...
base_category_wer, tuned_category_wer = ..., ...
seen_terms, unseen_terms = ..., ...
recall = {
    "baseline_seen": ...,
    "tuned_seen": ...,
    "baseline_unseen": ...,
    "tuned_unseen": ...,
}

print(f"Test WER: {base_wer:.1%} -> {tuned_wer:.1%}")
for category in base_category_wer:
    print(f"  {category:>8}: {base_category_wer[category]:.1%} -> {tuned_category_wer[category]:.1%}")
for key, r in recall.items():
    print(f"  term recall, {key:>16}: {r['hits']}/{r['total']}" + (f" = {r['recall']:.0%}" if r["recall"] is not None else ""))

In [ ]:
assert len(tuned_hyps) == len(test_rows), "one transcript per test recording"
assert abs(base_wer - corpus_wer(test_refs, base_hyps)) < 1e-9 and abs(tuned_wer - corpus_wer(test_refs, tuned_hyps)) < 1e-9
assert set(base_category_wer) == set(tuned_category_wer) == {row["category"] for row in test_rows}, "one WER per category"
assert not set(seen_terms) & set(unseen_terms), "a term can't be both seen and unseen"
assert all(any(contains_term(text, t) for text in train_texts) for t in seen_terms), "every seen term must appear in a training sentence"
assert not any(any(contains_term(text, t) for text in train_texts) for t in unseen_terms), "no unseen term may appear in a training sentence"
assert all(any(contains_term(ref, t) for ref in test_refs) for t in seen_terms + unseen_terms), "keep only terms that appear in the test references"
assert set(recall) == {"baseline_seen", "tuned_seen", "baseline_unseen", "tuned_unseen"}
print("Correct.")

Which test transcripts changed? Reading them is the best way to understand what fine-tuning did, beyond the numbers.

In [ ]:
changed = [(ref, b, t) for ref, b, t in zip(test_refs, base_hyps, tuned_hyps) if b.strip() != t.strip()]
print(f"{len(changed)} of {len(test_refs)} test transcripts changed:\n")
for ref, b, t in changed[:12]:
    print(f"reference: {ref}\nbefore:    {b.strip()}\nafter:     {t.strip()}\n")

## Part 4: did it forget?

**Catastrophic forgetting** is when a model trained on new data gets worse at what it could already do. Each experiment measured its LibriSpeech WER after training; here you compare the chosen model with the original on the same clips. A small rise (say, 9.0% to 9.5%) is a fair price for a big gain on your voice. A large rise means the model over-specialized.

### TODO(you): LibriSpeech before and after

Compute `libri_before` (a fresh whisper-tiny.en) and `libri_after` (your `tuned_model`) on `libri_audios` and `libri_refs`. Each takes about 40 seconds.

In [ ]:
# TODO(you): LibriSpeech WER of the original and the chosen model.
libri_before = ...
libri_after = ...

print(f"LibriSpeech WER: {libri_before:.1%} -> {libri_after:.1%}")

In [ ]:
assert abs(libri_after - best["librispeech_wer"]) < 0.005, "this should match the LibriSpeech WER the experiment recorded"
if not SMOKE_TEST:
    assert 0.07 < libri_before < 0.11, "whisper-tiny.en scored about 9% on these clips in notebook 01"
print("Correct.")

## Part 5: how sure are you?

Your test set has about 42 sentences. With a different 42 sentences, both WERs would come out a bit different. So is the improvement real, or could it be luck in which sentences landed in the test set?

### The bootstrap

The **bootstrap** estimates that uncertainty from the data you already have:

1. Draw 42 sentences **with replacement** from your 42 test sentences, so some appear twice and some not at all. That's one plausible alternative test set.
2. On that sample, compute the baseline's WER and the tuned model's WER, and take the difference.
3. Repeat 2,000 times. The middle 95% of the differences, from the 2.5th to the 97.5th percentile, is a **95% confidence interval** for the change.

It's a **paired** bootstrap: each sample uses the same sentences for both models, because what you care about is the difference on the same speech.

If the whole interval is below zero, the tuned model is better on essentially every plausible test set: a robust result. If the interval includes zero, you can't claim an improvement.

### Making it fast

Corpus WER is total errors divided by total reference words. So compute each sentence's error count and word count once; after that, each bootstrap sample is just sums.

- `jiwer.process_words(reference, hypothesis)` aligns two strings and returns an object with `.substitutions`, `.deletions`, `.insertions`, and `.hits` (correct words), all ints. A sentence's errors are S + D + I, and its number of reference words is S + D + hits.
- `np.array(a_list)` turns a list of numbers into a NumPy array.
- `rng = np.random.default_rng(seed)` creates a NumPy random generator, and `rng.integers(0, n, size=n)` draws `n` random ints from 0 to n − 1, with replacement.
- With NumPy arrays `errors` and `words`, and an array of indices `idx`, `errors[idx]` picks those positions (repeats included), so `errors[idx].sum() / words[idx].sum()` is the WER of that sample.
- `np.percentile(values, [2.5, 97.5])` returns the two percentiles.

Normalize every string with `processor.tokenizer.normalize` before `process_words`, as always.

### TODO(you): write `bootstrap_wer_difference(references, hyps_a, hyps_b, n_samples=2000, seed=0)`

**Inputs:** three lists of str of the same length, the number of samples, and a seed.

**Output:** a tuple `(low, high)` of floats: the 95% confidence interval of WER(b) − WER(a). Negative means b is better.

**Steps:**
1. Normalize everything, then build three NumPy arrays: errors for a, errors for b, and reference word counts (the same for both).
2. Draw `n_samples` samples of indices, and for each one compute WER(b) − WER(a) on that sample.
3. Return the 2.5th and 97.5th percentiles as floats.

In [ ]:
# TODO(you): paired bootstrap confidence interval for WER(b) - WER(a).
def bootstrap_wer_difference(references, hyps_a, hyps_b, n_samples=2000, seed=0):
    ...


low, high = bootstrap_wer_difference(test_refs, base_hyps, tuned_hyps)
print(f"Change in test WER: {tuned_wer - base_wer:+.1%}, 95% confidence interval [{low:+.1%}, {high:+.1%}]")

In [ ]:
refs3 = ["a b c d", "e f g h", "i j k l"]
lo, hi = bootstrap_wer_difference(refs3, refs3, refs3)
assert lo == 0.0 and hi == 0.0, "identical hypotheses give a difference of exactly 0"
lo, hi = bootstrap_wer_difference(refs3, refs3, ["a b c x", "e f g x", "i j k x"])
assert abs(lo - 0.25) < 1e-9 and abs(hi - 0.25) < 1e-9, \
    "every sentence has exactly 1 error in 4 words, so the difference is +25% in every sample"
assert isinstance(low, float) and isinstance(high, float) and low <= high
print("Correct.")

## Part 6: the write-up

The next two cells save every number to `results/final.json` and draw the charts for your write-up, saved as `results/results.png`. `json.dumps(obj, indent=2)` turns a dict into nicely indented text.

In [ ]:
final = {
    "chosen_experiment": best_name,
    "n_test_recordings": len(test_rows),
    "test_wer": {"baseline": base_wer, "tuned": tuned_wer, "ci95_change": [low, high]},
    "test_category_wer": {"baseline": base_category_wer, "tuned": tuned_category_wer},
    "term_recall": {key: {k: v for k, v in r.items() if k != "misses"} for key, r in recall.items()},
    "librispeech_wer": {"baseline": libri_before, "tuned": libri_after},
    "base_en_test_wer": baseline["test_wer"]["openai/whisper-base.en"],
}
save_json(final, RESULTS_DIR / ("final_smoke.json" if SMOKE_TEST else "final.json"))
print(json.dumps(final, indent=2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

categories = list(base_category_wer)
x = np.arange(len(categories) + 1)
axes[0].bar(x - 0.2, [base_wer] + [base_category_wer[c] for c in categories], width=0.4, label="whisper-tiny.en")
axes[0].bar(x + 0.2, [tuned_wer] + [tuned_category_wer[c] for c in categories], width=0.4, label="fine-tuned")
axes[0].set_xticks(x, ["all"] + categories)
axes[0].set_title("Test WER on my voice")
axes[0].legend()

groups = [("seen", "baseline_seen", "tuned_seen"), ("unseen", "baseline_unseen", "tuned_unseen")]
x = np.arange(len(groups))
axes[1].bar(x - 0.2, [recall[b]["recall"] or 0 for _, b, _ in groups], width=0.4, label="whisper-tiny.en")
axes[1].bar(x + 0.2, [recall[t]["recall"] or 0 for _, _, t in groups], width=0.4, label="fine-tuned")
axes[1].set_xticks(x, [f"{name} terms" for name, _, _ in groups])
axes[1].set_ylim(0, 1)
axes[1].set_title("Key-term recall on the test set")
axes[1].legend()

axes[2].bar(["whisper-tiny.en", "fine-tuned"], [libri_before, libri_after], color=["tab:blue", "tab:orange"])
axes[2].set_title("LibriSpeech WER (general speech)")

for ax in axes:
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
plt.tight_layout()
plt.savefig(RESULTS_DIR / ("results_smoke.png" if SMOKE_TEST else "results.png"), dpi=150)
plt.show()

### TODO(you): your story

Fill in the sentence with your numbers, then add a short paragraph: what improved most, what didn't, what the experiments taught you, and what you'd try next. This paragraph goes in your README, and it's what you'll talk through in an interview.

> On my held-out test set of N recordings, fine-tuning whisper-tiny.en cut WER from **X%** to **Y%** (95% confidence interval for the change: [a, b] percentage points), and key-term recall on terms seen in training rose from **P%** to **Q%**. WER on general speech (LibriSpeech) went from **9.0%** to **Z%**. The best variant was **...**, chosen on the validation set before looking at the test set.

*(Your paragraph here.)*

## What's next

Commit your results: `results/` is small and goes to GitHub, while `models/` stays on your laptop.

    git add -A
    git commit -m "Notebook 04: honest evaluation"
    git push

An optional next step: convert the fine-tuned model to CTranslate2 format, plug it into flowlite, and measure latency.

### Check your understanding

1. Why choose the winner on validation instead of test? What would your reported number mean if you had picked it on test?
2. Explain your confidence interval to a product manager who doesn't know statistics.
3. What did fine-tuning do for unseen terms compared with seen terms? What does that tell you about what it learned?
4. Which variant forgot the least on LibriSpeech, and why might that be?
5. What does the encoder learn, and what does the decoder learn? What kind of improvement do you give up by freezing the encoder?
6. Why does LoRA need a higher learning rate than full fine-tuning?
7. How would you serve personalized Whisper models to thousands of users with low latency? Think about per-user adapters, merging, and batching requests from different users together.